# GraphRAG-Bench (ICLR 2026)

**Paper:** [When to use Graphs in RAG](https://arxiv.org/abs/2506.05690) — Xiang et al.  
**Repo / dataset:** [GraphRAG-Bench/GraphRAG-Benchmark](https://github.com/GraphRAG-Bench/GraphRAG-Benchmark) · HF `GraphRAG-Bench/GraphRAG-Bench`

Central question: *when do graph structures help vs vanilla vector RAG?*

| Level | Task | Paper finding |
|-------|------|---------------|
| 1 | **Fact Retrieval** | Basic RAG ≈ / beats GraphRAG (Obs.1) |
| 2 | **Complex Reasoning** | GraphRAG wins (Obs.2) |
| 3 | **Contextual Summarize** | GraphRAG wins |
| 4 | **Creative Generation** | GraphRAG wins |

This notebook builds a **Novel-split mini subset** (one book, 2 questions × 4 levels), runs our local methods, and demos the ID→question map + metric autopsy.

In [ ]:
# ── Config ─────────────────────────────────────────────────
N_PER_TYPE = 2          # questions per GraphRAG-Bench level
SOURCE = "Novel-4128"  # matches results_graphrag_bench/  # or None to auto-pick smallest full-coverage novel
RUN_BENCHMARK = False   # True to re-run; False reads results_graphrag_bench/
REUSE_INDEXES = True
METHODS = [
    "semantic_rag",       # basic RAG — should compete on Fact Retrieval
    "rerank_semantic",
    "hybrid_rag",         # vec + GraphRAG local
    "frontier_rag",       # adaptive + CRAG escalate
    # "lazygraph_rag",    # GraphRAG fast/basic (slow first index)
]
# ───────────────────────────────────────────────────────────

import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

from rag_benchmark import (
    BenchmarkConfig,
    BenchmarkRunner,
    create_tracked_client,
    build_graphrag_bench_subset,
)
from rag_benchmark.charts import METHOD_LABELS, plot_dashboard, print_leaderboard
from rag_benchmark.decision_playbook import build_decision_artifacts
from rag_benchmark.engineering import (
    build_engineering_scorecard,
    print_engineering_briefing,
    save_engineering_scorecard,
)

RESULTS = PROJECT_ROOT / "results_graphrag_bench"
RESULTS.mkdir(parents=True, exist_ok=True)
print(f"Project={PROJECT_ROOT}")
print(f"Results={RESULTS}")

## 1. Build GraphRAG-Bench Novel subset

Downloads Novel corpus + questions from the official repo (cached under `data/graphrag_bench_cache/`), picks one book, and writes:
- `data/corpus_graphrag_bench/*.txt`
- `data/qa/graphrag_bench_eval.json`
- `results/graphrag_bench_question_catalog.csv`

In [ ]:
built = build_graphrag_bench_subset(
    project_root=PROJECT_ROOT,
    n_per_type=N_PER_TYPE,
    source=SOURCE,
)
meta = built["meta"]
display(Markdown("### Subset meta"))
display(pd.Series(meta).to_frame("value"))

config = BenchmarkConfig.from_yaml(PROJECT_ROOT)
config.project_root = PROJECT_ROOT
config.corpus_dir = built["corpus_dir"]
config.qa_path = built["qa_path"]
config.semantic_collection = "graphrag_bench_semantic"
config.graph_workspace = PROJECT_ROOT / "graphrag_workspaces" / "graphrag_bench"
config.lazy_workspace = PROJECT_ROOT / "graphrag_workspaces" / "graphrag_bench_lazy"
config.reuse_indexes = REUSE_INDEXES
config.graph_indexing_method = "fast"
config.semantic_top_k = 8
config.max_documents = 10_000
config.results_dir = lambda: RESULTS  # type: ignore[method-assign]

print(f"Corpus docs: {len(list(Path(built['corpus_dir']).glob('*.txt')))}")
print(f"QA: {built['qa_path']}")

## Demo: question ID → actual question

GraphRAG-Bench IDs look like `Novel-…`. This catalog maps **Q1…** to task level, question, and gold.

In [ ]:
import json

qa = json.loads(Path(built["qa_path"]).read_text(encoding="utf-8"))
catalog = pd.read_csv(PROJECT_ROOT / "results" / "graphrag_bench_question_catalog.csv")
display(catalog)

print("\n── Full questions ──")
for i, q in enumerate(qa):
    print(f"Q{i+1} [{q['graphrag_bench_type']}]")
    print(f"  id:   {q['id']}")
    print(f"  Q:    {q['question']}")
    print(f"  Gold: {q['expected_answer'][:200]}{'…' if len(str(q['expected_answer']))>200 else ''}\n")

## 2. Run methods (or load prior results)

Paper expectation on this ladder:
- **Fact Retrieval** → semantic / rerank competitive
- **Complex / Summarize / Creative** → hybrid / FrontierRAG should pull ahead

In [ ]:
results = []
if RUN_BENCHMARK:
    runner = BenchmarkRunner(config, create_tracked_client(config))
    results = runner.run_all(methods=METHODS)

    accuracy_df = BenchmarkRunner.to_accuracy_frame(results)
    qa_by_id = {q["id"]: q for q in qa}
    accuracy_df["graphrag_bench_type"] = accuracy_df["question_id"].map(
        lambda i: qa_by_id.get(i, {}).get("graphrag_bench_type", "")
    )
    summary_df = BenchmarkRunner.to_summary_frame(results, config)
    scenario_df = BenchmarkRunner.to_scenario_frame(results)
    token_df = BenchmarkRunner.to_token_frame(results, config)
    latency_df = BenchmarkRunner.to_latency_frame(results)

    accuracy_df.to_csv(RESULTS / "accuracy_results.csv", index=False)
    summary_df.to_csv(RESULTS / "summary.csv", index=False)
    scenario_df.to_csv(RESULTS / "scenario_results.csv", index=False)
    token_df.to_csv(RESULTS / "token_results.csv", index=False)
    latency_df.to_csv(RESULTS / "latency_results.csv", index=False)

    by_type = (
        accuracy_df.groupby(["graphrag_bench_type", "method"])["composite_score"]
        .mean()
        .reset_index()
        .sort_values(["graphrag_bench_type", "composite_score"], ascending=[True, False])
    )
    by_type.to_csv(RESULTS / "by_question_type.csv", index=False)

    plot_dashboard(RESULTS)
    scorecard = build_engineering_scorecard(summary_df, scenario_df, accuracy_df)
    save_engineering_scorecard(scorecard, RESULTS)
    build_decision_artifacts(RESULTS, Path(built["qa_path"]))
    print_leaderboard(RESULTS)
    print_engineering_briefing(scorecard)
else:
    print("RUN_BENCHMARK=False — loading", RESULTS)
    assert (RESULTS / "accuracy_results.csv").exists(), (
        "No results yet. Set RUN_BENCHMARK=True or run:\n"
        "  PYTHONPATH=src python scripts/run_graphrag_bench.py"
    )

## 3. When do graphs help? (by GraphRAG-Bench level)

This is the demo slide: winners per task type on *your* local run.

In [ ]:
acc = pd.read_csv(RESULTS / "accuracy_results.csv")
if "graphrag_bench_type" not in acc.columns:
    qa_by_id = {q["id"]: q for q in json.loads(Path(built["qa_path"]).read_text())}
    acc["graphrag_bench_type"] = acc["question_id"].map(
        lambda i: qa_by_id.get(i, {}).get("graphrag_bench_type", "")
    )

by_type = (
    acc.groupby(["graphrag_bench_type", "method"])[[
        "composite_score", "llm_judge_score", "token_f1", "exact_match", "contains_answer"
    ]]
    .mean()
    .reset_index()
)
by_type["label"] = by_type["method"].map(lambda m: METHOD_LABELS.get(m, m))

order = [
    "Fact Retrieval",
    "Complex Reasoning",
    "Contextual Summarize",
    "Creative Generation",
]
winners = []
for qt in order:
    sub = by_type[by_type.graphrag_bench_type == qt]
    if sub.empty:
        continue
    best = sub.loc[sub.composite_score.idxmax()]
    winners.append({
        "task_level": qt,
        "winner": best["label"],
        "composite": round(float(best.composite_score), 3),
        "judge": round(float(best.llm_judge_score), 3),
        "em": round(float(best.exact_match), 3),
        "paper_hint": (
            "vector OK" if qt == "Fact Retrieval" else "graphs help"
        ),
    })

display(Markdown("### Winners by task level"))
display(pd.DataFrame(winners))

pivot = by_type.pivot_table(
    index="label", columns="graphrag_bench_type", values="composite_score"
)
pivot = pivot.reindex(columns=[c for c in order if c in pivot.columns])
display(Markdown("### Composite by method × task level"))
display(pivot.round(3))

ax = pivot.T.plot(kind="bar", figsize=(10, 4.5), rot=20)
ax.set_ylabel("Mean composite (0–1)")
ax.set_title("GraphRAG-Bench Novel subset — quality by task level")
ax.set_ylim(0, 1)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

## 4. Metric autopsy (why EM looks “broken”)

Same story as Hotpot: **Creative Generation** and long summarize answers crush EM/F1 while LLM judge / contains stay higher. Prefer judge+contains for generative levels; EM for Fact Retrieval.

In [ ]:
import matplotlib.pyplot as plt

metric_means = (
    acc.groupby("method")[["llm_judge_score", "contains_answer", "token_f1", "exact_match"]]
    .mean()
    .rename(index=lambda m: METHOD_LABELS.get(m, m))
    .sort_values("llm_judge_score")
)
display(metric_means.round(3))

fig, ax = plt.subplots(figsize=(10, 4.5))
metric_means.plot(kind="barh", ax=ax)
ax.set_xlim(0, 1)
ax.set_xlabel("Mean score")
ax.set_title("GraphRAG-Bench subset — judge vs lexical metrics")
plt.tight_layout()
plt.show()

n = len(acc)
hi = int(((acc.llm_judge_score >= 0.5) & (~acc.exact_match.astype(bool))).sum())
display(Markdown(
    f"**{hi}/{n}** rows: judge ≥ 0.5 but EM = 0 — generative answers ≠ gold spans."
))

## 5. Engineering scorecard + charts

In [ ]:
print_leaderboard(RESULTS)
brief = RESULTS / "engineering_briefing.md"
if brief.exists():
    display(Markdown(brief.read_text(encoding="utf-8")))
dash = RESULTS / "benchmark_dashboard.png"
if dash.exists():
    display(Image(filename=str(dash)))
cheat = RESULTS / "routing_cheatsheet.md"
if cheat.exists():
    display(Markdown(cheat.read_text(encoding="utf-8")[:2500]))

## 6. How this relates to the full GraphRAG-Bench paper

| Paper component | This harness |
|-----------------|--------------|
| Novel + Medical corpora | Demo uses **Novel** mini-subset (one book) |
| 4 task levels | Same labels; n=2 each for local demo |
| Indexing / retrieval / generation evals | We score generation (judge+EM+F1+contains) + tokens/latency |
| Official eval scripts | Their `Evaluation/` (ROUGE, coverage, faithfulness) — next upgrade |
| Leaderboards | [graphrag-bench.github.io](https://graphrag-bench.github.io) |

**CLI:** `PYTHONPATH=src python scripts/run_graphrag_bench.py semantic_rag,hybrid_rag,frontier_rag 2`

HotpotQA notebook history remains in git; this notebook is the GraphRAG-Bench demo path.